# BigEarthNet to Petastorm Native Format

This notebook:
1. Reads metadata from S3
2. Stratified samples by split (train/val/test)
3. Loads S2, S1, and reference map images as NumPy arrays
4. Writes to **Petastorm native format** using `materialize_dataset()`

resource: https://www.databricks.com/notebooks/simple-aws/petastorm.html

## 1. Setup Environment



In [28]:
import boto3
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import rasterio
from rasterio.enums import Resampling
from io import BytesIO
import json
import os

from petastorm.codecs import NdarrayCodec
from petastorm.etl.dataset_metadata import materialize_dataset
from petastorm.unischema import Unischema, UnischemaField, dict_to_spark_row
import numpy as np

#from pyspark.sql import SparkSession

In [ ]:
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .appName("BigEarth") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.access.key", "") \
    .config("spark.hadoop.fs.s3a.secret.key", "") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "50000") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "30000") \
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "30000000") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory",  "4g")\
    .config("spark.executor.core",  "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

## 2. Configuration

In [30]:
# S3 paths
BUCKET = 'ubs-homes'
METADATA_KEY = 'erasmus/ola/bigearthnet/metadata/metadata_updated.parquet'

# Output path (local first for testing, then S3)
cwd = os.getcwd()
OUTPUT_PATH = f'file://{cwd}/petastorm_output' # Local
#OUTPUT_PATH = 's3a://ubs-homes/erasmus/ola/bigearthnet/petastorm_0.0001_1'  # S3

# Band configuration
S2_BANDS = ['B02', 'B03', 'B04', 'B08']  # 4 bands (10m resolution)
S1_BANDS = ['VV', 'VH']
TARGET_SIZE = (120, 120)

# Sampling config
SUBSET_ENABLED = True
SUBSET_FRACTION = 0.0001      # 1% of data
SUBSET_SEED = 42

## 3. Define Petastorm Schema - s2 and s1 have different stack 

In [31]:
# Note: s2_image shape matches number of S2_BANDS (4 bands)
#BigEarthNetSchema = Unischema('BigEarthNetSchema', [
 #   UnischemaField('s2_image', np.float32, (120, 120, 6), NdarrayCodec(), False),
  #  UnischemaField('s1_image', np.float32, (120, 120, 2), NdarrayCodec(), False),
   # UnischemaField('label_map', np.float32, (120, 120), NdarrayCodec(), False),
#])


## 3. Define Petastorm Schema -  for s1 and s2 into 1 image (120,120,6)

In [32]:
# stack or image into 1 
BigEarthNetSchema = Unischema('BigEarthNetSchema', [
    UnischemaField('image', np.float32, (120, 120, 6), NdarrayCodec(), False),
    UnischemaField('label_map', np.float32, (120, 120), NdarrayCodec(), False),
])

print("Schema defined!")

Schema defined!


## 4. Helper Functions

In [33]:
# S3 Client
s3 = boto3.client('s3')


def parse_s3_path(s3_path):
    """Parse s3://bucket/key into (bucket, key)"""
    path = s3_path.replace("s3://", "")
    bucket = path.split("/")[0]
    key = "/".join(path.split("/")[1:])
    return bucket, key


def load_band_from_s3(s3_path, target_size=(120, 120)):
    """Load a single TIF band from S3 and resample to target size"""
    bucket, key = parse_s3_path(s3_path)
    response = s3.get_object(Bucket=bucket, Key=key)
    file_bytes = BytesIO(response['Body'].read())
    
    with rasterio.open(file_bytes) as src:
        data = src.read(
            1,
            out_shape=target_size,
            resampling=Resampling.bilinear
        )
    return data.astype(np.float32)


def load_s2_patch(s2_folder_path, patch_id):
    """Load Sentinel-2 bands and stack into single array"""
    bands = []
    for band in S2_BANDS:
        band_path = f"{s2_folder_path}{patch_id}_{band}.tif"
        bands.append(load_band_from_s3(band_path, TARGET_SIZE))
    return np.stack(bands, axis=-1)  # (120, 120, 4)


def load_s1_patch(s1_folder_path, s1_name):
    """Load Sentinel-1 VV and VH bands"""
    bands = []
    for band in S1_BANDS:
        band_path = f"{s1_folder_path}{s1_name}_{band}.tif"
        bands.append(load_band_from_s3(band_path, TARGET_SIZE))
    return np.stack(bands, axis=-1)  # (120, 120, 2)


def load_reference_map(ref_folder_path, patch_id):
    """Load reference map (label) with NEAREST resampling for categorical data"""
    ref_path = f"{ref_folder_path}{patch_id}_reference_map.tif"
    bucket, key = parse_s3_path(ref_path)
    response = s3.get_object(Bucket=bucket, Key=key)
    file_bytes = BytesIO(response['Body'].read())
    
    with rasterio.open(file_bytes) as src:
        data = src.read(
            1,
            out_shape=TARGET_SIZE,
            resampling=Resampling.nearest  # Use nearest for categorical labels!
        )
    return data.astype(np.float32)


print("Helper functions defined!")

Helper functions defined!


## 5. Load Metadata

In [34]:
# Load metadata from S3
print(f"Loading metadata from s3://{BUCKET}/{METADATA_KEY}")
response = s3.get_object(Bucket=BUCKET, Key=METADATA_KEY)
metadata_df = pq.read_table(BytesIO(response['Body'].read())).to_pandas()

print(f"Loaded {len(metadata_df)} rows")
print(f"Columns: {metadata_df.columns.tolist()}")
metadata_df.head()

Loading metadata from s3://ubs-homes/erasmus/ola/bigearthnet/metadata/metadata_updated.parquet
Loaded 480038 rows
Columns: ['patch_id', 'labels', 'split', 'country', 's1_name', 's2v1_name', 'contains_seasonal_snow', 'contains_cloud_or_shadow', 's1_path', 's2_path', 'ref_path']


,patch_id,labels,split,country,s1_name,s2v1_name,contains_seasonal_snow,contains_cloud_or_shadow,s1_path,s2_path,ref_path
0,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Arable land, Broad-leaved forest, Mixed fores...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_26_57,S2A_MSIL2A_20170613T101031_26_57,False,False,s3://ubs-datasets/bigearthnet/BigEarthNet-S1/S...,s3://ubs-datasets/bigearthnet/BigEarthNet-S2/S...,s3://ubs-datasets/bigearthnet/Reference_Maps/S...
1,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Arable land, Broad-leaved forest, Inland wate...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_27_55,S2A_MSIL2A_20170613T101031_27_55,False,False,s3://ubs-datasets/bigearthnet/BigEarthNet-S1/S...,s3://ubs-datasets/bigearthnet/BigEarthNet-S2/S...,s3://ubs-datasets/bigearthnet/Reference_Maps/S...
2,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Arable land, Broad-leaved forest, Coniferous ...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_27_56,S2A_MSIL2A_20170613T101031_27_56,False,False,s3://ubs-datasets/bigearthnet/BigEarthNet-S1/S...,s3://ubs-datasets/bigearthnet/BigEarthNet-S2/S...,s3://ubs-datasets/bigearthnet/Reference_Maps/S...
3,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Broad-leaved forest, Complex cultivation patt...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_27_57,S2A_MSIL2A_20170613T101031_27_57,False,False,s3://ubs-datasets/bigearthnet/BigEarthNet-S1/S...,s3://ubs-datasets/bigearthnet/BigEarthNet-S2/S...,s3://ubs-datasets/bigearthnet/Reference_Maps/S...
4,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Broad-leaved forest, Complex cultivation patt...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_27_58,S2A_MSIL2A_20170613T101031_27_58,False,False,s3://ubs-datasets/bigearthnet/BigEarthNet-S1/S...,s3://ubs-datasets/bigearthnet/BigEarthNet-S2/S...,s3://ubs-datasets/bigearthnet/Reference_Maps/S...


## 6. Stratified Sampling

In [35]:
if SUBSET_ENABLED:
    metadata_df = metadata_df.groupby('split', group_keys=False).apply(
        lambda x: x.sample(frac=SUBSET_FRACTION, random_state=SUBSET_SEED)
    )
    print(f"Sampled {len(metadata_df)} rows")
    print("\nDistribution by split:")
    print(metadata_df['split'].value_counts())

Sampled 48 rows

Distribution by split:
split
train         24
test          12
validation    12
Name: count, dtype: int64


/var/folders/2m/h3tmqc297_qfhms3m1j64ygc0000gn/T/ipykernel_14782/1383495351.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  metadata_df = metadata_df.groupby('split', group_keys=False).apply(


## 7. Process Patches into Records - 2 images

In [36]:
''' # Process patches
records = []
total = len(metadata_df)

for idx, row in metadata_df.iterrows():
    try:
        s2_image = load_s2_patch(row['s2_path'], row['patch_id'])
        s1_image = load_s1_patch(row['s1_path'], row['s1_name'])
        label_map = load_reference_map(row['ref_path'], row['patch_id'])
        
        records.append({
            's2_image': s2_image,      # NumPy array
            's1_image': s1_image,      # NumPy array
            'label_map': label_map,    # NumPy array
        })
        
        if len(records) % 100 == 0:
            print(f"Processed {len(records)}/{total}...")
            
    except Exception as e:
        print(f"Error {row['patch_id']}: {e}")
        continue

print(f"\nTotal records: {len(records)}")

'''

' # Process patches\nrecords = []\ntotal = len(metadata_df)\n\nfor idx, row in metadata_df.iterrows():\n    try:\n        s2_image = load_s2_patch(row[\'s2_path\'], row[\'patch_id\'])\n        s1_image = load_s1_patch(row[\'s1_path\'], row[\'s1_name\'])\n        label_map = load_reference_map(row[\'ref_path\'], row[\'patch_id\'])\n        \n        records.append({\n            \'s2_image\': s2_image,      # NumPy array\n            \'s1_image\': s1_image,      # NumPy array\n            \'label_map\': label_map,    # NumPy array\n        })\n        \n        if len(records) % 100 == 0:\n            print(f"Processed {len(records)}/{total}...")\n            \n    except Exception as e:\n        print(f"Error {row[\'patch_id\']}: {e}")\n        continue\n\nprint(f"\nTotal records: {len(records)}")\n\n'

## 7. Process Patches into Records - 1 images

In [37]:
records = []
total = len(metadata_df)

for idx, row in metadata_df.iterrows():
    try:
        s2_image = load_s2_patch(row['s2_path'], row['patch_id']).astype("float32")  # (120,120,4)
        s1_image = load_s1_patch(row['s1_path'], row['s1_name']).astype("float32")  # (120,120,2)
        label_map = load_reference_map(row['ref_path'], row['patch_id']).astype("float32")  # (120,120)

        image = np.concatenate([s2_image, s1_image], axis=-1)  # (120,120,6)

        records.append({
            "image": image,
            "label_map": label_map
        })

        if len(records) % 100 == 0:
            print(f"Processed {len(records)}/{total}...")

    except Exception as e:
        print(f"Error {row['patch_id']}: {e}")
        continue

print(f"\nTotal records: {len(records)}")



Total records: 48


## 8. Verify One Record

In [38]:
# Check first record
sample = records[0]

print(f"image shape: {sample['image'].shape}, dtype: {sample['image'].dtype}")
#print(f"s1_image shape: {sample['s1_image'].shape}, dtype: {sample['s1_image'].dtype}")
print(f"label_map shape: {sample['label_map'].shape}, dtype: {sample['label_map'].dtype}")

image shape: (120, 120, 6), dtype: float32
label_map shape: (120, 120), dtype: float32


## 9. Write with Petastorm 

In [39]:

print(f"Writing {len(records)} records to {OUTPUT_PATH}")

# Convert to Spark Rows using Petastorm helper
rows_rdd = spark.sparkContext.parallelize(records) \
    .map(lambda x: dict_to_spark_row(BigEarthNetSchema, x))

# Write with Petastorm
with materialize_dataset(spark, OUTPUT_PATH, BigEarthNetSchema, row_group_size_mb=256):
    spark.createDataFrame(rows_rdd, BigEarthNetSchema.as_spark_schema()) \
        .write \
        .mode('overwrite') \
        .parquet(OUTPUT_PATH)

print("Done!")

Writing 48 records to file:///Users/omowonuolaakintola/Software/Personal Programming/UBS/Big Data/DL/petastorm_output


26/01/14 15:02:25 WARN TaskSetManager: Stage 2 contains a task of very large size (1585 KiB). The maximum recommended task size is 1000 KiB.
/Users/omowonuolaakintola/anaconda3/envs/petastorm_env/lib/python3.10/site-packages/petastorm/fs_utils.py:88: FutureWarning: pyarrow.localfs is deprecated as of 2.0.0, please use pyarrow.fs.LocalFileSystem instead.
  self._filesystem = pyarrow.localfs
/Users/omowonuolaakintola/anaconda3/envs/petastorm_env/lib/python3.10/site-packages/petastorm/fs_utils.py:89: FutureWarning: pyarrow.localfs is deprecated as of 2.0.0, please use pyarrow.fs.LocalFileSystem instead.
  self._filesystem_factory = lambda: pyarrow.localfs
/Users/omowonuolaakintola/anaconda3/envs/petastorm_env/lib/python3.10/site-packages/petastorm/utils.py:104: FutureWarning: 'ParquetDataset.fs' attribute is deprecated as of pyarrow 5.0.0 and will be removed in a future version. Specify 'use_legacy_dataset=False' while constructing the ParquetDataset, and then use the '.filesystem' attrib

Done!


/Users/omowonuolaakintola/anaconda3/envs/petastorm_env/lib/python3.10/site-packages/petastorm/fs_utils.py:89: FutureWarning: pyarrow.localfs is deprecated as of 2.0.0, please use pyarrow.fs.LocalFileSystem instead.
  self._filesystem_factory = lambda: pyarrow.localfs
/Users/omowonuolaakintola/anaconda3/envs/petastorm_env/lib/python3.10/site-packages/petastorm/fs_utils.py:89: FutureWarning: pyarrow.localfs is deprecated as of 2.0.0, please use pyarrow.fs.LocalFileSystem instead.
  self._filesystem_factory = lambda: pyarrow.localfs
/Users/omowonuolaakintola/anaconda3/envs/petastorm_env/lib/python3.10/site-packages/petastorm/fs_utils.py:89: FutureWarning: pyarrow.localfs is deprecated as of 2.0.0, please use pyarrow.fs.LocalFileSystem instead.
  self._filesystem_factory = lambda: pyarrow.localfs
/Users/omowonuolaakintola/anaconda3/envs/petastorm_env/lib/python3.10/site-packages/petastorm/fs_utils.py:89: FutureWarning: pyarrow.localfs is deprecated as of 2.0.0, please use pyarrow.fs.LocalF

In [40]:
spark.stop()
print("Spark stopped")

Spark stopped


---
# Reading the Petastorm Dataset

Now that data is saved in Petastorm native format, reading is much simpler - no manual decoding needed!

In [41]:
from petastorm import make_reader

# Read with Petastorm - arrays are automatically decoded!
with make_reader(OUTPUT_PATH) as reader:
    for row in reader:
        #print(f"patch_id: {row.patch_id}")
        #print(f"split: {row.split}")
        print(f"image shape: {row.image.shape}")   # Already NumPy array!
        #print(f"s1_image shape: {row.s1_image.shape}")   # Already NumPy array!
        print(f"label_map shape: {row.label_map.shape}") # Already NumPy array!
        break

s2_image shape: (120, 120, 6)
label_map shape: (120, 120)
Cache cleanup complete.


/Users/omowonuolaakintola/anaconda3/envs/petastorm_env/lib/python3.10/site-packages/petastorm/etl/dataset_metadata.py:402: FutureWarning: Specifying the 'metadata_nthreads' argument is deprecated as of pyarrow 8.0.0, and the argument will be removed in a future version
  dataset = pq.ParquetDataset(path_or_paths, filesystem=fs, validate_schema=False, metadata_nthreads=10)
/Users/omowonuolaakintola/anaconda3/envs/petastorm_env/lib/python3.10/site-packages/petastorm/etl/dataset_metadata.py:362: FutureWarning: 'ParquetDataset.common_metadata' attribute is deprecated as of pyarrow 5.0.0 and will be removed in a future version.
  if not dataset.common_metadata:
/Users/omowonuolaakintola/anaconda3/envs/petastorm_env/lib/python3.10/site-packages/petastorm/etl/dataset_metadata.py:368: FutureWarning: 'ParquetDataset.common_metadata' attribute is deprecated as of pyarrow 5.0.0 and will be removed in a future version.
  dataset_metadata_dict = dataset.common_metadata.metadata
/Users/omowonuolaaki

## Tensorflow DataLoader

In [42]:
'''import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import models, layers


def get_model():
  model = models.Sequential()
  model.add(layers.Conv2D(32, kernel_size=(3, 3),
                          activation='relu',
                          input_shape=(120, 120, 6)))
  model.add(layers.Conv2D(64, (3, 3), activation='relu'))
  model.add(layers.MaxPooling2D(pool_size=(2, 2)))
  model.add(layers.Dropout(0.25))
  model.add(layers.Flatten())
  model.add(layers.Dense(128, activation='relu'))
  model.add(layers.Dropout(0.5))
  model.add(layers.Dense(10, activation='softmax'))
  return model'''

"import tensorflow as tf\nfrom tensorflow import keras\nfrom tensorflow.keras import models, layers\n\n\ndef get_model():\n  model = models.Sequential()\n  model.add(layers.Conv2D(32, kernel_size=(3, 3),\n                          activation='relu',\n                          input_shape=(120, 120, 6)))\n  model.add(layers.Conv2D(64, (3, 3), activation='relu'))\n  model.add(layers.MaxPooling2D(pool_size=(2, 2)))\n  model.add(layers.Dropout(0.25))\n  model.add(layers.Flatten())\n  model.add(layers.Dense(128, activation='relu'))\n  model.add(layers.Dropout(0.5))\n  model.add(layers.Dense(10, activation='softmax'))\n  return model"